# Cleaning V3 Sample 1000 Smoke Validation

This Notebook verifies the refactored cleaning planner, scheduler, evaluator, state, and export flow on the default MinIO `sample_1000` dataset.

## 0. Bootstrap repo root

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if not (repo_root / "pyproject.toml").exists():
    raise RuntimeError("cannot locate repository root")

src_root = repo_root / "src"
for path in [repo_root, src_root]:
    path_text = str(path)
    if path_text in sys.path:
        sys.path.remove(path_text)
sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(src_root))

print("repo_root:", repo_root)
print("src_root:", src_root)


repo_root: /home/wuchaoli/codespace/ImageGallery/.worktrees/cleaning-planner-scheduler-refactor
src_root: /home/wuchaoli/codespace/ImageGallery/.worktrees/cleaning-planner-scheduler-refactor/src


## 1. Imports and runtime paths

In [2]:
import json

import pandas as pd

from image_gallery.cleaning import BasicCleaner
from notebooks._helpers.cleaning_configs import get_cleaning_v3_first_batch_operator_configs
from notebooks._helpers.datasets import (
    get_default_minio_sample_1000_raw_path,
    load_default_minio_sample_1000_dataset,
    load_default_minio_sample_1000_frame,
)
from notebooks._helpers.paths import get_notebook_library_root, reset_output_dir

pd.set_option("display.max_columns", 80)

RUN_ROOT = get_notebook_library_root("cleaning_v3_sample_1000")
RUN_OUTPUT_DIR = reset_output_dir(RUN_ROOT / "run")
EXPORT_DIR = reset_output_dir(RUN_ROOT / "exports")
PREVIEW_DIR = reset_output_dir(RUN_ROOT / "preview")

raw_path = get_default_minio_sample_1000_raw_path()
raw_frame = load_default_minio_sample_1000_frame()

print("raw_path:", raw_path)
print("run_output_dir:", RUN_OUTPUT_DIR)
print("export_dir:", EXPORT_DIR)
print("preview_dir:", PREVIEW_DIR)


raw_path: /home/wuchaoli/codespace/ImageGallery/.worktrees/cleaning-planner-scheduler-refactor/notebooks/.importers_test_library/default_minio_dataset/sample_1000/raw.parquet
run_output_dir: /home/wuchaoli/codespace/ImageGallery/.worktrees/cleaning-planner-scheduler-refactor/notebooks/.operators_test_library/cleaning_v3_sample_1000/run
export_dir: /home/wuchaoli/codespace/ImageGallery/.worktrees/cleaning-planner-scheduler-refactor/notebooks/.operators_test_library/cleaning_v3_sample_1000/exports


## 2. Validate raw dataset

In [3]:
required_raw_columns = {"image_id", "image_uri"}
missing_raw_columns = sorted(required_raw_columns.difference(raw_frame.columns))
if missing_raw_columns:
    raise AssertionError(f"missing raw dataset columns: {missing_raw_columns}")
if raw_frame["image_uri"].isna().any():
    raise AssertionError("raw dataset contains empty image_uri values")

print("raw_rows:", len(raw_frame))
print("raw_columns:", raw_frame.columns.tolist())
raw_frame[["image_id", "image_uri"]].head()


raw_rows: 1000
raw_columns: ['image_id', 'source_uri', 'source_type', 'source_file_name', 'storage_name', 'image_uri', 'import_status', 'imported_at', 'schema_version', 'tags', 'file_size_bytes', 'image_content_hash', 'file_extension', 'mime_type', 'image_format', 'width', 'height', 'pixel_count', 'aspect_ratio', 'orientation', 'channels', 'color_mode', 'has_alpha', 'animated', 'frame_count', 'icc_profile_present', 'dpi_x', 'dpi_y', 'exif_orientation', 'exif_datetime', 'camera_make', 'camera_model', 'gps_present', 'gps_latitude', 'gps_longitude']
                               image_id  \
0  001ab42f-e567-4912-bcb6-ea5bcdbc8280   
1  00a6d3a7-c561-493c-b0d6-978c876f6b94   
2  016cf5f4-4d51-456d-8acb-805874c0e259   
3  01fcce40-1bbd-4395-bc31-c89accd1e1d6   
4  020706b0-26eb-48ec-ac98-7ee816cea90c   

                                           image_uri  
0  s3://test/images/raw/2026-07-06/shard_002/001a...  
1  s3://test/images/raw/2026-07-06/shard_002/00a6...  
2  s3://test/images/raw

## 3. Load storage-backed dataset

In [4]:
dataset = load_default_minio_sample_1000_dataset()

sample_images = []
for image_uri in raw_frame["image_uri"].astype(str).head(3):
    image = dataset.read_image(image_uri)
    sample_images.append(
        {
            "image_uri": image_uri,
            "size": image.size,
            "format": image.format,
        }
    )

pd.DataFrame(sample_images)


                                           image_uri        size format
0  s3://test/images/raw/2026-07-06/shard_002/001a...  (960, 540)    PNG
1  s3://test/images/raw/2026-07-06/shard_002/00a6...  (960, 540)    PNG
2  s3://test/images/raw/2026-07-06/shard_002/016c...  (640, 480)    PNG


## 4. Compile and inspect execution plan

In [5]:
operator_configs = get_cleaning_v3_first_batch_operator_configs()
operator_names = [next(iter(item.keys())) for item in operator_configs]

cleaner = BasicCleaner(operator_configs, output_dir=RUN_OUTPUT_DIR)
plan_frame = cleaner.plan()

expected_plan_columns = {
    "step_index",
    "computer_name",
    "execution_mode",
    "requested_parameters",
    "required_parameters",
    "produced_parameters",
    "upstream_computers",
}
missing_plan_columns = sorted(expected_plan_columns.difference(plan_frame.columns))
if missing_plan_columns:
    raise AssertionError(f"missing plan columns: {missing_plan_columns}")
if plan_frame.empty:
    raise AssertionError("compiled parameter plan is empty")

execution_modes = set(plan_frame["execution_mode"].astype(str))
required_modes = {"per_image", "dataset_aggregate"}
missing_modes = sorted(required_modes.difference(execution_modes))
if missing_modes:
    raise AssertionError(f"missing execution modes: {missing_modes}")

print("operator_names:", operator_names)
print("execution_modes:", sorted(execution_modes))
plan_frame


operator_names: ['format.decode_check', 'size.dimension_check', 'size.aspect_ratio_check', 'size.megapixel_check', 'quality.blur_check', 'quality.brightness_check', 'quality.contrast_check', 'content.blank_image_check', 'duplicate.exact_duplicate_check']
execution_modes: ['dataset_aggregate', 'per_image', 'table']
   step_index             computer_name     execution_mode  \
0           0       image_hash_computer          per_image   
1           1   image_metadata_computer          per_image   
2           2    image_quality_computer          per_image   
3           3    table_derived_computer              table   
4           4  duplicate_group_computer  dataset_aggregate   

                                requested_parameters required_parameters  \
0                                       content_hash                       
1                decode_error,decode_ok,height,width                       
2  blank_score,blur_score,brightness_score,contra...                       
3      

## 5. Run BasicCleaner

In [6]:
dataset = load_default_minio_sample_1000_dataset()
result = cleaner.run(dataset, progress="auto")
preview = result.preview(limit=5)
state_frame = result.state()
preview_html_path = result.preview_html(PREVIEW_DIR / "quality_blur.html", operator_name="quality.blur_check")

print("preview:", preview)
state_frame


preview: PreviewResult(total_count=1000, clean_count=998, review_count=0, dropped_count=2, restricted_count=0, operator_summary=                     operator_name  keep  review  drop  restricted
0              format.decode_check  1000       0     0           0
1             size.dimension_check  1000       0     0           0
2          size.aspect_ratio_check  1000       0     0           0
3             size.megapixel_check  1000       0     0           0
4               quality.blur_check  1000       0     0           0
5         quality.brightness_check  1000       0     0           0
6           quality.contrast_check  1000       0     0           0
7        content.blank_image_check  1000       0     0           0
8  duplicate.exact_duplicate_check   998       0     2           0, sample_rows=                               image_id  \
0  001ab42f-e567-4912-bcb6-ea5bcdbc8280   
1  00a6d3a7-c561-493c-b0d6-978c876f6b94   
2  016cf5f4-4d51-456d-8acb-805874c0e259   
3  01fcce40-1bbd-

## 6. Validate exported result views


In [7]:
parameter_table_path = result.export_table("parameter", RUN_OUTPUT_DIR / "parameter_table.parquet")
evaluation_table_path = result.export_table("evaluation", RUN_OUTPUT_DIR / "evaluation_table.parquet")
manifest_path = result.export_manifest(RUN_OUTPUT_DIR / "parameter_manifest.json")

for output_path in [parameter_table_path, evaluation_table_path, manifest_path, preview_html_path]:
    if not output_path.exists():
        raise AssertionError(f"missing output file: {output_path}")

parameter_table = pd.read_parquet(parameter_table_path)
evaluation_table = pd.read_parquet(evaluation_table_path)
manifest_payload = json.loads(manifest_path.read_text(encoding="utf-8"))
parameter_manifest = manifest_payload["parameter_manifest"]
operator_outputs = manifest_payload["operator_outputs"]

required_parameter_columns = {
    "decode_ok",
    "decode_error",
    "width",
    "height",
    "aspect_ratio",
    "megapixels",
    "blur_score",
    "brightness_score",
    "contrast_score",
    "blank_score",
    "content_hash",
    "exact_duplicate_group_id",
    "exact_duplicate_count",
}
missing_parameter_columns = sorted(required_parameter_columns.difference(parameter_table.columns))
if missing_parameter_columns:
    raise AssertionError(f"missing parameter columns: {missing_parameter_columns}")

required_evaluation_columns = {"image_id", "image_uri", "final_action", "final_reason"}
missing_evaluation_columns = sorted(required_evaluation_columns.difference(evaluation_table.columns))
if missing_evaluation_columns:
    raise AssertionError(f"missing evaluation columns: {missing_evaluation_columns}")

if len(parameter_table) != len(raw_frame):
    raise AssertionError("parameter table row count does not match raw frame")
if len(evaluation_table) != len(raw_frame):
    raise AssertionError("evaluation table row count does not match raw frame")
if result.status() != "completed":
    raise AssertionError(f"unexpected cleaner state: {result.status()}")

print("parameter_table_shape:", parameter_table.shape)
print("evaluation_table_shape:", evaluation_table.shape)
print("preview_html_path:", preview_html_path)
print("manifest_parameters:", sorted(parameter_manifest))
print("operator_outputs:", sorted(operator_outputs))


run_dir: /home/wuchaoli/codespace/ImageGallery/.worktrees/cleaning-planner-scheduler-refactor/notebooks/.operators_test_library/cleaning_v3_sample_1000/run/20260708T035619088562Z-379da5aa
parameter_table_shape: (1000, 16)
evaluation_table_shape: (1000, 31)
manifest_parameters: ['aspect_ratio', 'blank_score', 'blur_score', 'brightness_score', 'content_hash', 'contrast_score', 'decode_error', 'decode_ok', 'exact_duplicate_count', 'exact_duplicate_group_id', 'height', 'megapixels', 'width']


## 7. Summary checks

In [8]:
action_counts = (
    evaluation_table["final_action"]
    .value_counts(dropna=False)
    .rename_axis("final_action")
    .reset_index(name="count")
)

operator_status_counts = (
    state_frame["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="count")
)

display(action_counts)
display(operator_status_counts)
state_frame


  final_action  count
0         keep    998
1         drop      2
      status  count
0  completed      9
                     operator_name     status  processed_count  skipped_count  \
0              format.decode_check  completed             1000              0   
1             size.dimension_check  completed             1000              0   
2          size.aspect_ratio_check  completed             1000              0   
3             size.megapixel_check  completed             1000              0   
4               quality.blur_check  completed             1000              0   
5         quality.brightness_check  completed             1000              0   
6           quality.contrast_check  completed             1000              0   
7        content.blank_image_check  completed             1000              0   
8  duplicate.exact_duplicate_check  completed             1000              0   

   failed_count message  
0             0    None  
1             0    None  
2    

## 8. Inspect duplicate or dropped rows

In [9]:
analysis_columns = [
    "image_id",
    "image_uri",
    "final_action",
    "final_reason",
    "exact_duplicate_action",
    "exact_duplicate_reason",
    "exact_duplicate_group_id",
    "exact_duplicate_count",
]
analysis_columns = [column for column in analysis_columns if column in evaluation_table.columns]
analysis_frame = evaluation_table[analysis_columns].copy()

dropped_or_duplicate = analysis_frame.query(
    "final_action != 'keep' or exact_duplicate_count > 1",
    engine="python",
).head(10)

dropped_or_duplicate


                                 image_id  \
389  628a0308-d53e-4f49-9cb6-1b845a881e60   
499  7f911237-d2a4-4633-8335-903cade9bb7e   
935  ef4df5c7-947b-4101-92e1-91a727fe72ee   

                                             image_uri final_action  \
389  s3://test/images/raw/2026-07-06/shard_001/628a...         keep   
499  s3://test/images/raw/2026-07-06/shard_001/7f91...         drop   
935  s3://test/images/raw/2026-07-06/shard_001/ef4d...         drop   

                                          final_reason exact_duplicate_action  \
389                                                                      keep   
499  duplicate in group exact-fa26bfc141ea569915967...                   drop   
935  duplicate in group exact-fa26bfc141ea569915967...                   drop   

                                exact_duplicate_reason  \
389                                                      
499  duplicate in group exact-fa26bfc141ea569915967...   
935  duplicate in group exact-fa26b

## 9. Validate exports

In [10]:
full_export = result.export("full", str(EXPORT_DIR / "full.parquet")).to_frame()
clean_export = result.export("clean", str(EXPORT_DIR / "clean.parquet")).to_frame()
dropped_export = result.export("dropped", str(EXPORT_DIR / "dropped.parquet")).to_frame()

action_counts_map = evaluation_table["final_action"].value_counts(dropna=False).to_dict()
expected_clean_count = int(action_counts_map.get("keep", 0))
expected_dropped_count = int(action_counts_map.get("drop", 0))

if len(full_export) != len(evaluation_table):
    raise AssertionError("full export row count mismatch")
if len(clean_export) != expected_clean_count:
    raise AssertionError("clean export row count mismatch")
if len(dropped_export) != expected_dropped_count:
    raise AssertionError("dropped export row count mismatch")

print("full_export_rows:", len(full_export))
print("clean_export_rows:", len(clean_export))
print("dropped_export_rows:", len(dropped_export))
print("PASS: cleaning v3 planner/scheduler smoke validation completed")


full_export_rows: 1000
clean_export_rows: 998
dropped_export_rows: 2
PASS: cleaning v3 planner/scheduler smoke validation completed
